In [1]:
from spin_lattices import SquareLattice, KagomeLattice, SpinLattice
from spin_systems import (
    heisenberg,
    zero_sector_basis,
    ground_state_basis,
    spin_system,
    no_symmetries_basis,
    SpinSystem,
)
import pandas as pd
import numpy as np
import sympy as sp
import numba
import numpy.typing as npt
from misc_utils import spin_inv

In [3]:
lattice = KagomeLattice(2, 2)
system_nosym = spin_system(
    heisenberg(lattice, J1=1, J2=1),
    basis=no_symmetries_basis(),
    ground_state_cache_dir=None,
)
system_ground_state_basis = spin_system(
    heisenberg(lattice, J1=1, J2=1),
    basis=ground_state_basis(),
    ground_state_cache_dir=None,
)
assert (
    np.isclose(system_nosym.ground_energy,
    system_ground_state_basis.ground_energy
))

states = system_nosym.basis.states

2024-07-08 21:59:57.767 | DEBUG    | spin_systems:_find_cached_eigenstate:191 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 22:00:03.775 | DEBUG    | spin_systems:_find_cached_eigenstate:191 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 22:00:09.814 | DEBUG    | spin_systems:_find_cached_eigenstate:191 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 22:00:15.831 | DEBUG    | spin_systems:_find_cached_eigenstate:191 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 22:00:21.828 | DEBUG    | spin_systems:_find_cached_eigenstate:191 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 22:00:27.875 | DEBUG    | spin_systems:_find_cached_eigenstate:191 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 22:00:33.890 | DEBUG    | spin_systems:_find_cached_eigenstate:191 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 22:00:39.893 | D

In [9]:
states[:10]

array([ 63,  95, 111, 119, 123, 125, 126, 159, 175, 183], dtype=uint64)

In [30]:
def state_info(
        self, states: npt.NDArray[np.uint64]
    ) -> tuple[
        npt.NDArray[np.uint64], npt.NDArray[np.complex128], npt.NDArray[np.float64]
    ]:
        states_unpacked = self.lattice.unpack_configurations(states)
        perm_group_size = len(self.basis.symmetries)
        orbit = np.empty((perm_group_size, len(states)), dtype=np.uint64)
        spin_inversion = self.basis.spin_inversion
        for i, (permutation, _) in enumerate(self.basis.symmetries):
            orbit[i] = self.lattice.pack_configurations(
                states_unpacked[:, (permutation).array_form]
            )
        if spin_inversion is not None:
            orbit = np.concatenate([orbit, spin_inv(orbit, self.lattice.number_spins)])

        
        fix_size = (orbit[0] == orbit).sum(axis=0)
        n_orb = orbit.shape[0] // fix_size

        reprs = orbit.min(axis=0)
        is_repr = orbit == reprs

        g_idxs = np.argmin(orbit, axis=0)
        all_characters = np.array(
            [
                sp.exp(2 * sp.pi * sp.I * self.basis.symmetries[idx][1]).evalf()
                for idx in range(perm_group_size)
            ],
            dtype=np.complex128,
        )

        if spin_inversion is not None:
            all_characters = np.concatenate([all_characters, spin_inversion * all_characters])
        
        all_characters = all_characters.reshape(-1, 1)


        print(f"{all_characters.shape=}")
        print(f"{is_repr.shape=}")

        characters_matrix = np.where(is_repr, np.broadcast_to(all_characters, is_repr.shape), np.nan)

        if spin_inversion is not None:
            spin_invs, g_idxs = np.divmod(g_idxs, len(self.basis.symmetries))

        characters = np.nanmean(characters_matrix, axis=0)
        if spin_inversion is not None:
            characters *= np.where(spin_invs, spin_inversion, 1)

        
        norms = 1 / np.sqrt(n_orb)

        return reprs, characters, norms


In [31]:
state_info(system_ground_state_basis, states[:10])

all_characters.shape=(96, 1)
is_repr.shape=(96, 10)


(array([ 63,  95, 111, 119, 123, 119, 126, 111,  95, 126], dtype=uint64),
 array([ 0.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j,  1.+0.j,
        -1.+0.j, -1.+0.j, -1.+0.j]),
 array([0.28867513, 0.10206207, 0.10206207, 0.10206207, 0.14433757,
        0.10206207, 0.14433757, 0.10206207, 0.10206207, 0.14433757]))

In [6]:
np.allclose(
    system_nosym.get_ground_state_coeffs(states, apply_symmetries=False)[:10],
    -system_ground_state_basis.get_ground_state_coeffs(states, apply_symmetries=True)[:10])

True

array([ 0.        ,  0.00183765,  0.00204682,  0.01452042, -0.01726599,
       -0.01452042,  0.01338152, -0.00204682, -0.00183765, -0.01338152])

In [2]:
system = spin_system(heisenberg(KagomeLattice(2, 4), J2=1), ground_state_basis())

2024-07-08 16:46:01.413 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 16:46:08.002 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 16:46:14.600 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 16:46:21.167 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 16:46:27.687 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 16:46:34.251 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 16:46:40.787 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.
2024-07-08 16:46:47.351 | D

In [156]:
len(system.basis.symmetries)

128

In [187]:
def spin_inv(x: npt.NDArray[np.uint64], number_spins: int) -> npt.NDArray[np.uint64]:
    return (2 ** number_spins - 1) ^ x

In [164]:
spin_inv(5, 3)

2

In [201]:
states = np.array([667634, 15626253, 30587, 55223])


def state_info(
    system: SpinSystem, states: npt.NDArray[np.uint64]
) -> tuple[npt.NDArray[np.uint64], npt.NDArray[np.complex128], npt.NDArray[np.float64]]:
    states_unpacked = system.lattice.unpack_configurations(states)
    perm_group_size = len(system.basis.symmetries)
    orbit = np.empty((perm_group_size, len(states)), dtype=np.uint64)
    spin_inversion = system.basis.spin_inversion
    for i, (permutation, _) in enumerate(system.basis.symmetries):
        orbit[i] = system.lattice.pack_configurations(
            states_unpacked[:, (permutation).array_form]
        )
    if spin_inversion is not None:
        orbit = np.concatenate([orbit, spin_inv(orbit, system.lattice.number_spins)])

    g_idxs = np.argmin(orbit, axis=0)
    reprs = orbit.min(axis=0)

    if spin_inversion is not None:
        spin_invs, g_idxs = np.divmod(g_idxs, len(system.basis.symmetries))

    characters = np.array(
        [
            sp.exp(2 * sp.pi * sp.I * system.basis.symmetries[idx][1]).evalf()
            for idx in g_idxs
        ],
        dtype=np.complex128,
    )
    if spin_inversion is not None:
        characters *= np.where(spin_invs, spin_inversion, 1)

    fix_size = (orbit[0] == orbit).sum(axis=0)
    n_orb = orbit.shape[0] // fix_size
    norms = 1 / np.sqrt(n_orb)

    return reprs, characters, norms

In [119]:
system_nosym = spin_system(heisenberg(KagomeLattice(2, 4), J2=1), no_symmetries_basis())

In [120]:
system_nosym.get_ground_state_coeffs(states, apply_symmetries=False)

2024-07-08 20:22:50.326 | DEBUG    | spin_systems:_find_cached_eigenstate:189 - Using cached version of eigenvalues / eigenstates up to 1.


array([-3.36272656e-08,  3.47667742e-08,  3.47667742e-08, -1.87797689e-08])

In [196]:
reprs, characters, norms = state_info(system, states)
system.get_ground_state_coeffs(reprs, apply_symmetries=False) * characters * norms

array([ 3.36272656e-08, -3.47667742e-08, -3.47667742e-08,  1.87797689e-08])